# Introducción a Tensorflow 2


In [32]:
import time
import pandas as pd
import numpy as np
import pandas as pd
import tensorflow as tf
from matplotlib import pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
print(tf.__version__)

2.21.0


## Conjunto de datos
Para comenzar, utilizaremos un clásico conjunto de datos denominado [MNIST](http://yann.lecun.com/exdb/mnist/), formado por un conjunto de imágenes de dígitos manuscritos
![images](https://storage.googleapis.com/tfds-data/visualization/fig/mnist-3.0.1.png)

In [33]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

El conjunto de entrenamiento está formado por 60.000 imágenes de 28x28, mientras que el de test lo forman 10.000 imágenes. El problema de clasificación consiste en distinguir entre los 10 dígitos diferentes

In [34]:
print("Datos de entrenamiento", x_train.shape)
print("Datos de test", x_test.shape)
print("Primeros dígitos: ", y_train[:5])

Datos de entrenamiento (60000, 28, 28)
Datos de test (10000, 28, 28)
Primeros dígitos:  [5 0 4 1 9]


## Pre-procesamiento
Como suele ser habitual, es conveniente realizar algún tipo de procesamiento antes de que los datos sean tratados por el modelo. En este caso, haremos un pequeño procesamiento que consiste en escalar los valores entre 0 y 1. Para ello, bastará con dividir por 255, ya que cada pixel de las imágenes de entrada viene representado por un entero entre 0 y 255 (8 bits) indicando su tonalidad de gris.

In [35]:
x_train = x_train / 255.0
x_test = x_test / 255.0
x_train.dtype

dtype('float64')

## Modelo
Para crear un modelo, primero tenemos que definir la arquitectura neuronal. El caso más sencillo es definir una estructura secuencial de capas, donde la primera actúa de capa de entrada y la última de salida del modelo. Cada capa tendrá una serie de parámetros dependiendo del tipo, habitualmente habrá que especificar el tamaño (número de neuronas), función de activación, etc.

In [36]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28,28)),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(10, activation='softmax')
])

/home/miguel/Desktop/uma-26-27-1/.venv/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## Entrenamiento
Antes de entrenar, hay que compilar el modelo indicando:
- La **regla de aprendizaje** (*optimizer*) que define el algoritmo para actualizar los pesos sinápticos de la red
- La **función objetivo** o de pérdida (*loss*), que es optimizada por la regla de aprendizaje
- La **métrica** o medida de evaluación (*metrics*) utilizada para evaluar el modelo entrenado

In [37]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

Tras compilar el modelo, el entrenamiento se define mediante los siguientes parámetros:
- **Épocas** (*epochs*). Número de veces que se le muestra el conjunto de entrenamiento al modelo
- Tamaño de **lote** (*batch_size*). Número de instancias o patrones que se muestran al modelo para actualizar los pesos sinápticos
- Conjunto de **validación** (*validation_spli*). Datos utilizados para medir el rendimiento durante el entrenamiento

En este caso, se ha optado por los valores por defecto

In [38]:
h=model.fit(x_train, y_train, epochs=5)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 973us/step - accuracy: 0.9272 - loss: 0.2552
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9681 - loss: 0.1094  
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 968us/step - accuracy: 0.9785 - loss: 0.0741
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9842 - loss: 0.0542  
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9890 - loss: 0.0399   


## Evaluación
Una vez entrenada la red neuronal, prodremos obtener las medidas de rendimiento espcificadas en la etapa anterior, tanto para el conjunto de entrenamiento como para el conjunto de test

In [39]:
model.evaluate(x_train, y_train)
model.evaluate(x_test, y_test)

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 638us/step - accuracy: 0.9906 - loss: 0.0321
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 683us/step - accuracy: 0.9763 - loss: 0.0795


[0.07945146411657333, 0.9763000011444092]

---

In [40]:
def entrenar_evaluar(opt, lr, batch_size, epochs, x_tr, y_tr, x_te, y_te):
    tf.keras.utils.set_random_seed(42)

    match opt.lower(): # elegimos el optimizador de uno de los siguientes
        case "adam": optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
        case "rmsprop": optimizer = tf.keras.optimizers.RMSprop(learning_rate=lr)
        case "sgd": optimizer = tf.keras.optimizers.SGD(learning_rate=lr)
        case "sgd_momentum": optimizer = tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
        case _: raise ValueError("Optimizador incorrecto")

    model1 = tf.keras.models.Sequential([ # creamos la red
        tf.keras.layers.Flatten(input_shape=(28,28)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

    model1.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    t_ini = time.time() # calculamos el tiempo exacto del entrenamiento
    h = model1.fit(x_tr, y_tr, epochs=epochs, batch_size=batch_size)
    t_total = time.time() - t_ini

    prob_tr = model1.predict(x_tr, batch_size=batch_size) # obtenemos las salidas del modelo (lista de probabilidades)
    prob_te = model1.predict(x_te, batch_size=batch_size)
    pred_tr = np.argmax(prob_tr, axis=1) # la predicccion es la salida con mayor probabilidad
    pred_te = np.argmax(prob_te, axis=1)

    acc_tr = accuracy_score(y_tr, pred_tr) # calculamos el accuracy
    acc_te = accuracy_score(y_te, pred_te)

    num_err_tr = int(sum(pred_tr != y_tr)) # numero de errores
    num_err_te = int(sum(pred_te != y_te))

    prec_te = precision_score(y_te, pred_te, average="macro")
    rec_te = recall_score(y_te, pred_te, average="macro")
    f1_te = f1_score(y_te, pred_te, average="macro")
    auc_te = roc_auc_score(y_te, prob_te, multi_class="ovr")

    cm_tr = confusion_matrix(y_tr, pred_tr) # creamos las matrices de confusion
    cm_te = confusion_matrix(y_te, pred_te)

    metricas = {
        "optimizador": opt,
        "learning_rate": lr,
        "batch_size": batch_size,
        "epochs": epochs,
        "tiempo_entrenamiento(s)": round(t_total, 2),
        "train_accuracy": round(acc_tr, 4),
        "test_accuracy": round(acc_te, 4),
        "errores_train": f"{num_err_tr}/{len(y_tr)}",
        "errores_test": f"{num_err_te}/{len(y_te)}",
        "precision": round(prec_te, 4),
        "recall": round(rec_te, 4),
        "f1_score": round(f1_te, 4),
        "auc_roc": round(auc_te, 4),
    }

    return metricas, model1, cm_tr, cm_te, pred_te, h

def resumen(cm_te, h):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    cmd = ConfusionMatrixDisplay(cm_te)
    cmd.plot(ax=axes[0])

    axes[1].plot(h.history["loss"], label="loss", color="b")
    axes[1].plot(h.history["accuracy"], label="accuracy", color="g")
    axes[1].set_xlabel("epochs")
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

## Experimento 1



In [41]:
met1, mod1, cm_tr1, cm_te1, pred_te1, h1 = entrenar_evaluar(
    opt="adam",
    lr=0.001,
    batch_size=32,
    epochs=5,
    x_tr=x_train,
    y_tr=y_train,
    x_te=x_test,
    y_te=y_test,
)

/home/miguel/Desktop/uma-26-27-1/.venv/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9276 - loss: 0.2530
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9670 - loss: 0.1112
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9775 - loss: 0.0754
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9836 - loss: 0.0547
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9880 - loss: 0.0403  
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 444us/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 551us/step


In [42]:
display(pd.DataFrame(met1))
resumen(cm_te1, h1)

ValueError: If using all scalar values, you must pass an index